In [ ]:
## IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN DE ENTORNO ##
import pandas as pd
import numpy as np
from faker import Faker
from google.cloud import bigquery
from google.oauth2 import service_account
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
import random

# Cargar variables de entorno
load_dotenv()

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
CREDENTIALS_PATH = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Inicializar cliente y Faker
credentials = service_account.Credentials.from_service_account_file(CREDENTIALS_PATH)
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
fake = Faker(['es_ES']) # Datos localizados en España para mayor realismo

print(f"Entorno listo para generar datos en {PROJECT_ID}.{DATASET_ID}")

In [ ]:
## GENERACIÓN DE CATEGORÍAS Y PRODUCTOS ##
# 1. CATEGORÍAS
cat_names = ["Smartphones", "Laptops", "Audio", "Wearables", "Periféricos", "Gaming", "Accesorios"]
categories = []
for i, name in enumerate(cat_names):
    categories.append({
        "category_id": i + 1,
        "name": name,
        "description": f"Productos de la categoría {name}",
        "active": True,
        "created_at": datetime(2023, 1, 1)
    })

df_categories = pd.DataFrame(categories)

# 2. PRODUCTOS (70 productos)
products = []
for i in range(1, 71):
    cat = random.choice(categories)
    cost = round(random.uniform(10.0, 800.0), 2)
    margin = random.uniform(1.2, 1.5) # Margen de beneficio del 20% al 50%
    products.append({
        "product_id": i,
        "category_id": cat["category_id"],
        "sku": f"TECH-{i:04d}",
        "name": f"Producto {cat['name']} {fake.word().capitalize()}",
        "description": fake.sentence(),
        "sale_price": round(cost * margin, 2),
        "cost_price": cost,
        "stock_quantity": random.randint(0, 100),
        "active": True,
        "created_at": datetime(2023, 1, 5),
        "updated_at": None
    })

df_products = pd.DataFrame(products)
print(f"Generados {len(df_categories)} categorías y {len(df_products)} productos.")

In [ ]:
## GENERACIÓN DE CLIENTES ##
# CLIENTES (500)
customers = []
channels = ['organic', 'paid_ads', 'social_media', 'referral', 'email', 'other']

for i in range(1, 501):
    customers.append({
        "customer_id": i,
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "email": fake.unique.email(),
        "phone": fake.phone_number(),
        "country": "Spain",
        "city": fake.city(),
        "acquisition_channel": random.choice(channels),
        "registered_at": fake.date_time_between(start_date='-2y', end_date='-1y')
    })

df_customers = pd.DataFrame(customers)
print(f"Generados {len(df_customers)} clientes.")

In [ ]:
## GENERACIÓN DE PEDIDOS Y LÍNEAS DE PEDIDO ##
# PEDIDOS (2000) y LÍNEAS DE PEDIDO
orders = []
order_items = []
order_item_id_counter = 1

statuses = ['delivered', 'delivered', 'delivered', 'shipped', 'confirmed', 'cancelled', 'returned']

for i in range(1, 2001):
    customer = random.choice(customers)
    order_date = fake.date_time_between(start_date=customer["registered_at"], end_date='now')
    status = random.choice(statuses)
    
    orders.append({
        "order_id": i,
        "customer_id": customer["customer_id"],
        "order_status": status,
        "shipping_address": fake.address(),
        "shipping_city": customer["city"],
        "shipping_country": "Spain",
        "shipping_postal_code": fake.postcode(),
        "shipping_cost": round(random.uniform(0, 15), 2),
        "ordered_at": order_date,
        "shipped_at": order_date + timedelta(days=1) if status in ['shipped', 'delivered'] else None,
        "delivered_at": order_date + timedelta(days=3) if status == 'delivered' else None,
        "cancelled_at": order_date + timedelta(hours=5) if status == 'cancelled' else None
    })

    # Líneas de pedido (entre 1 y 5 productos por pedido, promedio ~2.25)
    num_items = random.randint(1, 5)
    selected_products = random.sample(products, num_items)
    
    for prod in selected_products:
        qty = random.randint(1, 3)
        order_items.append({
            "order_item_id": order_item_id_counter,
            "order_id": i,
            "product_id": prod["product_id"],
            "quantity": qty,
            "unit_price_at_purchase": prod["sale_price"],
            "unit_cost_at_purchase": prod["cost_price"],
            "discount_amount": 0.0,
            "line_total": round(qty * prod["sale_price"], 2)
        })
        order_item_id_counter += 1

df_orders = pd.DataFrame(orders)
df_order_items = pd.DataFrame(order_items)
print(f"Generados {len(df_orders)} pedidos con {len(df_order_items)} líneas de detalle.")

In [ ]:
## GENERACIÓN DE PAGOS Y VALORACIONES ##
# 1. PAGOS (1 por pedido)
payments = []
for i, row in df_orders.iterrows():
    # Sumar el total de las líneas de este pedido
    order_total = df_order_items[df_order_items['order_id'] == row['order_id']]['line_total'].sum()
    total_with_shipping = round(order_total + row['shipping_cost'], 2)
    
    status = 'completed' if row['order_status'] in ['delivered', 'shipped'] else 'pending'
    if row['order_status'] == 'cancelled': status = 'failed'
    if row['order_status'] == 'returned': status = 'refunded'

    payments.append({
        "payment_id": i + 1,
        "order_id": row['order_id'],
        "payment_method": random.choice(['card', 'paypal', 'bank_transfer']),
        "payment_status": status,
        "amount": total_with_shipping,
        "paid_at": row['ordered_at'] + timedelta(minutes=5) if status != 'failed' else None,
        "transaction_reference": f"TRX-{fake.uuid4()[:8].upper()}"
    })

df_payments = pd.DataFrame(payments)

# 2. VALORACIONES (~35% de los productos entregados)
reviews = []
delivered_items = df_order_items[df_order_items['order_id'].isin(df_orders[df_orders['order_status'] == 'delivered']['order_id'])]
sampled_items = delivered_items.sample(frac=0.35)

for i, (_, item) in enumerate(sampled_items.iterrows()):
    reviews.append({
        "review_id": i + 1,
        "order_item_id": item['order_item_id'],
        "rating": random.choices([5, 4, 3, 2, 1], weights=[40, 30, 15, 10, 5])[0],
        "comment": fake.sentence() if random.random() > 0.3 else None,
        "reviewed_at": df_orders.loc[df_orders['order_id'] == item['order_id'], 'delivered_at'].iloc[0] + timedelta(days=random.randint(1, 7))
    })

df_reviews = pd.DataFrame(reviews)
print(f"Generados {len(df_payments)} pagos y {len(df_reviews)} valoraciones.")

In [ ]:
##CARGA DE DATOS A BIGQUERY##
# Función para cargar a BigQuery
def load_to_bq(df, table_id):
    full_table_path = f"{PROJECT_ID}.{DATASET_ID}.{table_id}"
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE") # Sobrescribe si ya hay datos
    job = client.load_table_from_dataframe(df, full_table_path, job_config=job_config)
    job.result()
    print(f"Cargadas {len(df)} filas en {full_table_path}")

# Ejecutar carga en orden de dependencias
load_to_bq(df_customers, "customers")
load_to_bq(df_categories, "categories")
load_to_bq(df_products, "products")
load_to_bq(df_orders, "orders")
load_to_bq(df_order_items, "order_items")
load_to_bq(df_payments, "payments")
load_to_bq(df_reviews, "reviews")

print("\n--- ¡PROCESO DE CARGA FINALIZADO CON ÉXITO! ---")